# 学習済みモデルの評価と同期動画
学習をやり直さず、Driveのモデルを30回評価し、最初の死亡前後の同期動画を自動作成します。
左は元のAtari映像、右はアーケード風の表示です。結果は日時別のフォルダへ保存します。
途中で止まっても各回の評価CSVは残ります。再実行すると新しい評価を始めます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q gymnasium ale-py numpy scipy pyyaml pillow imageio imageio-ffmpeg matplotlib

In [ ]:
from pathlib import Path
Path('hra').mkdir(exist_ok=True)

In [ ]:
%%writefile hra/__init__.py
"""HRA (Hybrid Reward Architecture, van Seijen et al. 2017) の Ms. Pac-Man 再現"""

In [ ]:
%%writefile hra/config.yaml
# HRA (van Seijen et al., NeurIPS 2017) Ms. Pac-Man 再現 設定ファイル (NFR-04)
#
# 各値の出典:
#   [本文]   論文本文       [付録A.x] 論文 supplementary (arXiv:1706.04208 の付録)
#   [仮]     論文に記載なし。実装上の暫定値 (要件定義書の未確定事項 U-xx)

experiment:
  name: auto                # auto: 設定から自動命名 (FR-43 設定ごとに結果を分ける)
  seed: 0                   # NFR-02
  episodes: 11000           # [本文] は5,000。途中の不具合・方式変更のため延長。8,000 から D 方式 (step_mode=cell, ram, α=1) で継続 (DECISIONS.md)
  checkpoint_every: 10      # NFR-01 何エピソードごとに学習状態を保存するか (Colab を細切れに使うため短め)
  log_every: 10

env:
  game: ALE/MsPacman-v5
  frameskip: 8              # 暫定。付録A.3はA3C節でありHRA固有の設定の根拠ではない
  repeat_action_probability: 0.0   # ALE決定的環境 (論文当時の ALE は sticky action なし)
  crop_top: 6               # [仮 U-01] 210x160 → 160x160 (エサの行 7〜165 がすべて入る位置)
  cell: 4                   # [本文] 4ピクセル精度 → 40x40
  ghost_frames: 3           # [仮 U-01] ゴーストは3フレーム周期で交互描画されるため直近3フレームを合成
  position_source: ram      # pixels: 画面の色から抽出 / ram: ALE の RAM から読む (RM-10)。cell と組み合わせて最良だった
  step_mode: cell           # frames: frameskip フレーム固定 / cell: 次のマスに入るまで同じ行動を続ける (DECISIONS.md)
  cell_substep: 2           # step_mode=cell のとき、マスの移動を確かめる間隔 (フレーム)
  cell_max_frames: 16       # step_mode=cell のとき、1ステップの上限フレーム数 (壁に向かって止まっている場合など)

reward:
  ghost_contact: -1000      # [本文] FR-10 残機を失ったときの整形報酬

gvf:
  alpha: 1.0                # [付録A.2]。step_mode=cell で遷移がほぼ決定的になったので論文どおり 1.0 (frames のときは 0.2 が良かった)
  gamma_score: 0.99         # [付録A.2/A.4] スコア系の頭が使うGVFの割引率
  gamma_ghost: 0.99         # [付録A.4] ゴースト系の頭が使うGVFの割引率 (独立にグリッドサーチされている)
  target: expected_sarsa    # FR-22 expected_sarsa: 式(8) ランダム方策の価値 [付録A.2] / q_learning: 式(7)

heads:
  # [付録A.2] 各オブジェクトの頭 = そのオブジェクト位置のGVF × 倍率
  pellet: 10
  power_pellet: 50
  fruit: 200
  blue_ghost: 1000
  ghost_linear: -1000       # 線形アグリゲーター時のゴースト倍率
  ghost_normalized: -10     # [付録A.2] 正規化アグリゲーター時のゴースト倍率
  snap_radius: 1            # [仮 U-03] オブジェクト位置にGVFがないとき、この距離(マス)以内の既知位置のGVFを使う
  heading_bonus: 0.0        # [論文にない工夫] 進行方向を含む行動へのボーナス (0 で論文どおり)。往復して止まる動きを防ぐ

aggregator:
  type: normalized          # FR-26 normalized / linear   ([本文] 最良は normalized)
  normalize: max            # [仮 U-04] "score heads を 0〜1 に正規化": max = 最大値で割る / minmax

exploration:
  diversification: true     # FR-33
  div_max: 20.0             # [付録A.2] U(0, 20)
  div_steps: 50             # [付録A.2] 最初の50ステップのみ
  count_based: true         # FR-33
  kappa: 0.1                # [仮 U-05] 式(11) Q = kappa * sqrt(N^(1/4) / n(s,a)) の係数 (論文に値なし)

executive_memory:
  enabled: false            # FR-37 (最大スコア実験のみ on。通常評価と分けて報告する: FR-36)
  value: 1.0e+6             # [仮 U-07] "very high value"

evaluation:
  noop_max: 30              # [仮 U-08] 固定開始評価の no-op 上限 (文献の慣例値)
  human_starts: null        # [仮 U-08] ランダム開始評価用の人間軌跡 (行動列 .npy) のパス。入手後に指定
  episodes: 10

In [ ]:
%%writefile hra/env.py
"""
Atari 2600 Ms. Pac-Man (ALE) 環境とオブジェクト抽出 (FR-01〜FR-07, FR-10, RM-10〜RM-12)

画面 → 160x160 に切り出し → オブジェクトの位置を 4ピクセル精度 (40x40 マス) に変換する。
位置の取り方は設定 env.position_source で切り替える (RM-12):
  pixels : 画面の色から抽出 (論文と同じ方針。具体的な方法は論文に記載なし = U-01)
  ram    : ALE の RAM から読む (RM-10。AtariARI の Ms. Pac-Man 注釈のアドレスを使用)

画面抽出の方法 (実機画面の観察にもとづく。決定記録 DECISIONS.md を参照):
  - Ms. Pac-Man と 4体のゴーストはそれぞれ固有の色を持つ
  - 赤以外のゴーストは3フレーム周期で1体ずつ交互に描画されるので、直近3フレームを合成する
  - エサは迷路の壁と同じ色の小さな塊。面の開始時に見えた塊の位置を「エサの候補地」として登録し、
    以後はその画素だけを見て残っているかを判定する (RM-11)
  - 青ゴースト(点滅時は白)は色で判定し、塊ごとに1体とみなす
  - 既知の色以外の塊をフルーツとみなす
"""
from collections import deque

import numpy as np
import gymnasium as gym
from scipy import ndimage


def _rgb(r, g, b):
    return (r << 16) | (g << 8) | b


C_PACMAN = _rgb(210, 164, 74)
C_GHOSTS = [_rgb(200, 72, 72), _rgb(198, 89, 179), _rgb(84, 184, 153), _rgb(180, 122, 48)]  # 赤,ピンク,水色,橙
C_BLUE   = [_rgb(66, 114, 194), _rgb(214, 214, 214)]   # 青ゴースト (通常 / 点滅時)

# AtariARI (Anand et al., 2019) の Ms. Pac-Man RAM 注釈
RAM_PLAYER = (10, 16)                                # (x, y)
RAM_GHOSTS = [(9, 15), (8, 14), (7, 13), (6, 12)]    # blinky(赤), pinky, inky(水色), sue(橙)
# RAM座標 → 画面座標 (切り出し前) の変換。実測で |誤差| ≤ 2px (95%点)
RAM_TO_PIX = (-8, 6)                                 # pix_x = ram_x - 8, pix_y = ram_y + 6

N_ACTIONS = 9            # [付録A.2] nine actions
NORTH, EAST, SOUTH, WEST = range(4)   # FR-05 向き
GRID = 40
SCREEN_H = 210
PELLET_ROW_CENTER = 7.5  # エサの行の中心 (切り出し前の画面で 7.5, 19.5, 31.5, ... 行目。実測)
PELLET_ROW_STEP = 12
SITE_SCAN_FRAMES = 800   # 面の開始からこのフレーム数の間、候補地を追加し続ける
                         # (パワーエサは開始前の待ち時間 約400フレームは描かれず、その後も点滅するため)


def _pack(frame):
    f = frame.astype(np.int32)
    return (f[..., 0] << 16) | (f[..., 1] << 8) | f[..., 2]


def _center(mask):
    ys, xs = np.nonzero(mask)
    if len(ys) == 0:
        return None
    # 外接矩形の中心 (口のアニメーションで重心が揺れにくい)
    return (ys.min() + ys.max()) / 2.0, (xs.min() + xs.max()) / 2.0


def _is_small(sl):
    return sl[0].stop - sl[0].start <= 4 and sl[1].stop - sl[1].start <= 4


class Objects:
    """1ステップ分の抽出結果。位置はすべて 40x40 のマス座標 (row, col)"""
    __slots__ = ("pacman", "direction", "ghosts", "blue_ghosts", "fruit",
                 "pellets", "power_pellets", "maze")

    def __init__(self):
        self.pacman = None
        self.direction = EAST
        self.ghosts = [None] * 4        # 通常ゴースト (1体ずつ別)
        self.blue_ghosts = [None] * 4   # 青ゴースト (通常ゴーストとは別オブジェクト)
        self.fruit = None
        self.pellets = []
        self.power_pellets = []
        self.maze = 0

    def to_channels(self):
        """FR-03〜05: 11枚の 40x40 2値チャンネルと、向きの one-hot(4)"""
        ch = np.zeros((11, GRID, GRID), dtype=np.uint8)

        def put(i, p):
            if p is not None:
                ch[i, p[0], p[1]] = 1
        put(0, self.pacman)
        for i, g in enumerate(self.ghosts):
            put(1 + i, g)
        for i, g in enumerate(self.blue_ghosts):
            put(5 + i, g)
        put(9, self.fruit)
        for p in self.pellets + self.power_pellets:
            put(10, p)
        direction = np.zeros(4, dtype=np.uint8)
        direction[self.direction] = 1
        return ch, direction


class MsPacmanEnv:
    """
    ALE の Ms. Pac-Man を frameskip フレームずつ進める。
    step() → (objects, raw_reward, shaped_reward, done, info)
      shaped_reward = 得点 + ゴースト接触時の -1000 (FR-10)
      info: frames, lives, life_lost, level_cleared, level, in_transition
    """

    def __init__(self, cfg, seed=0):
        import ale_py
        gym.register_envs(ale_py)
        e = cfg["env"]
        self.frameskip = e["frameskip"]
        self.crop_top = e["crop_top"]
        self.cell = e["cell"]
        self.ghost_frames = e["ghost_frames"]
        self.source = e.get("position_source", "pixels")
        self.step_mode = e.get("step_mode", "frames")
        self.cell_substep = e.get("cell_substep", 2)
        self.cell_max_frames = e.get("cell_max_frames", 16)
        self.objs = None
        self._recent = deque(maxlen=self.ghost_frames)
        self.ghost_penalty = cfg["reward"]["ghost_contact"]
        self._gym = gym.make(e["game"], frameskip=1,
                             repeat_action_probability=e["repeat_action_probability"],
                             full_action_space=False)
        self.ale = self._gym.unwrapped.ale
        self.action_set = self._gym.unwrapped._action_set
        assert len(self.action_set) == N_ACTIONS
        self.seed = seed
        self.maze_colors = []   # 壁の色 → 迷路ID (FR-06)
        self.pellet_sites = {}  # 迷路ID → {マス: (y, x, is_power)}

    # ── 画面・座標 ──
    def _screen(self):
        return _pack(self.ale.getScreenRGB()[self.crop_top:self.crop_top + 160])

    def _cell(self, yx):
        r = int(yx[0] // self.cell)
        c = int(yx[1] // self.cell)
        if not (0 <= r < GRID and 0 <= c < GRID):
            return None
        return (r, c)

    def _ram_cell(self, ram, xy_addr):
        x = int(ram[xy_addr[0]]) + RAM_TO_PIX[0]
        y = int(ram[xy_addr[1]]) + RAM_TO_PIX[1] - self.crop_top
        return self._cell((y, x))

    @staticmethod
    def _colors(s):
        vals, counts = np.unique(s[::2, ::2], return_counts=True)
        order = np.argsort(-counts)
        bg = int(vals[order[0]])
        wall = int(vals[order[1]]) if len(order) > 1 else None
        return bg, wall   # 最多色=背景、2番目=壁 (エサも同じ色)

    def _on_pellet_row(self, y_cropped):
        d = (y_cropped + self.crop_top - PELLET_ROW_CENTER) % PELLET_ROW_STEP
        return min(d, PELLET_ROW_STEP - d) <= 3

    def _is_pellet(self, sl):
        """エサ(縦2×横4)/パワーエサ(縦7×横4、点滅中は欠けることがある)の形で、中心がエサの行にある塊だけを
        候補地にする。ゴーストが壁に重なって分断された壁の切れ端を、エサと取り違えないため"""
        h, w = sl[0].stop - sl[0].start, sl[1].stop - sl[1].start
        return w == 4 and 2 <= h <= 7 and self._on_pellet_row((sl[0].start + sl[0].stop - 1) // 2)

    def _add_sites(self, screens):
        """画面に見えているエサの塊のうち、未登録のものを候補地に追加して返す。
        パワーエサは点滅するので、面の開始から SITE_SCAN_FRAMES フレームの間くり返し呼ぶ"""
        sites = self.pellet_sites[self.maze]
        mask = np.zeros(screens[-1].shape, dtype=bool)
        for s in screens:
            m = s == self.wall
            mask |= m
            if self._wall_always is None:
                self._wall_always, self._wall_ever = m.copy(), m.copy()
            else:
                self._wall_always &= m
                self._wall_ever |= m
        # 点滅している画素 = 一度は壁の色になったが、ずっとではない画素。
        # パワーエサは壁に接していることがある (右端) ので、壁と切り離すためにこちらからも探す
        blinking = self._wall_ever & ~self._wall_always
        slices = []
        for mk in (mask, blinking):
            lab, _ = ndimage.label(mk)
            slices += [sl for sl in ndimage.find_objects(lab) if sl is not None]
        new = set()
        for sl in slices:
            if not self._is_pellet(sl):
                continue
            cell = self._cell(((sl[0].start + sl[0].stop - 1) / 2, (sl[1].start + sl[1].stop - 1) / 2))
            if cell is not None and cell not in sites:
                h = sl[0].stop - sl[0].start
                # 判定用の画素 = 塊の中心 (縦2のエサは上の行)
                sites[cell] = ((sl[0].start + sl[0].stop - 1) // 2, (sl[1].start + sl[1].stop) // 2, h >= 3)
                new.add(cell)
        if new or not hasattr(self, "_site_cells"):
            self._site_cells = list(sites.keys())
            self._site_y = np.array([sites[c][0] for c in self._site_cells], dtype=np.int64)
            self._site_x = np.array([sites[c][1] for c in self._site_cells], dtype=np.int64)
        return new

    def _register_maze(self, screens):
        """面の開始時: 壁の色で迷路を判定し (FR-06)、エサの候補地を登録する (FR-07, RM-11)"""
        self.background, wall = self._colors(screens[-1])
        if wall not in self.maze_colors:
            self.maze_colors.append(wall)
        self.maze, self.wall = self.maze_colors.index(wall), wall
        self.pellet_sites.setdefault(self.maze, {})
        if hasattr(self, "_site_cells"):
            del self._site_cells
        self._wall_always = self._wall_ever = None
        self._add_sites(screens)
        self.level_frames = 0

    def load_sites(self, maze_colors, pellet_sites):
        """保存したエサの候補地を読み込む。形の条件を満たさない古い候補地 (壁の切れ端) は捨てる"""
        self.maze_colors = list(maze_colors)
        self.pellet_sites = {m: {c: v for c, v in sites.items() if self._on_pellet_row(v[0])}
                             for m, sites in pellet_sites.items()}
        dropped = sum(len(s) for s in pellet_sites.values()) - sum(len(s) for s in self.pellet_sites.values())
        if dropped:
            print(f"[修正] エサの候補地から壁の切れ端 {dropped} 個を除外しました")

    def _visible_pellets(self, screens):
        if not self._site_cells:
            return set()
        seen = np.zeros(len(self._site_cells), dtype=bool)
        for s in screens:
            seen |= s[self._site_y, self._site_x] == self.wall
        return {c for c, v in zip(self._site_cells, seen) if v}

    # ── オブジェクト抽出 ──
    def _extract_pixels(self, screens):
        s = screens[-1]
        pac = _center(s == C_PACMAN)
        pac = self._cell(pac) if pac is not None else None
        ghosts = [None] * 4
        for i, col in enumerate(C_GHOSTS):
            for sc in reversed(screens[-self.ghost_frames:]):
                c = _center(sc == col)
                if c is not None:
                    ghosts[i] = self._cell(c)
                    break
        return pac, ghosts

    def _extract_ram(self):
        ram = self.ale.getRAM()
        pac = self._ram_cell(ram, RAM_PLAYER)
        ghosts = [self._ram_cell(ram, a) for a in RAM_GHOSTS]
        return pac, ghosts

    def _blue_ghosts(self, screens):
        blue = np.zeros(screens[-1].shape, dtype=bool)
        for sc in screens[-self.ghost_frames:]:
            for col in C_BLUE:
                blue |= sc == col
        if not blue.any():
            return []
        lab, n = ndimage.label(blue)
        cs = ndimage.center_of_mass(blue, lab, range(1, n + 1))
        cells = [self._cell(p) for p in sorted(cs, key=lambda p: (p[1], p[0]))]
        return [c for c in cells if c is not None][:4]

    def _fruit(self, s):
        known = [self.background, self.wall, C_PACMAN, 0, *C_GHOSTS, *C_BLUE]
        other = ~np.isin(s, known)
        if other.sum() < 6:
            return None
        return self._cell(_center(other))

    def _make_objects(self, screens):
        o = Objects()
        o.maze = self.maze
        if self.source == "ram":
            pac, ghosts = self._extract_ram()
        else:
            pac, ghosts = self._extract_pixels(screens)
        blues = self._blue_ghosts(screens)
        if self.source == "ram" and blues:
            # RAM には青状態がないので、青い塊に近いゴーストを青ゴーストとして扱う
            for i, g in enumerate(ghosts):
                if g is not None and any(max(abs(g[0] - b[0]), abs(g[1] - b[1])) <= 2 for b in blues):
                    ghosts[i] = None
        o.pacman = pac
        o.ghosts = ghosts
        for i, b in enumerate(blues):
            o.blue_ghosts[i] = b
        o.fruit = self._fruit(screens[-1])

        # 向き (FR-05): 位置の変化から求める。止まっているときは直前の向きを保つ
        if pac is not None and self.prev_pac is not None and pac != self.prev_pac:
            dr, dc = pac[0] - self.prev_pac[0], pac[1] - self.prev_pac[1]
            if abs(dc) > GRID // 2:     # トンネルの左右ワープ
                dc = -dc
            if abs(dr) >= abs(dc):
                self.direction = SOUTH if dr > 0 else NORTH
            else:
                self.direction = EAST if dc > 0 else WEST
        if pac is not None:
            self.prev_pac = pac
        o.direction = self.direction

        sites = self.pellet_sites[self.maze]
        o.pellets = [c for c in self.remaining if not sites[c][2]]
        o.power_pellets = [c for c in self.remaining if sites[c][2]]
        return o

    # ── エピソード ──
    def reset(self, noops=0):
        self._gym.reset(seed=self.seed)
        self.seed += 1
        for _ in range(noops):
            self.ale.act(self.action_set[0])
        s = self._screen()
        self._recent.clear()
        self._recent.append(s)
        self._register_maze([s])
        self.level = 1
        self.lives = self.ale.lives()
        self.remaining = self._visible_pellets([s])
        self.frames = 0
        self.prev_pac = None
        self.direction = EAST
        self.in_transition = False
        self.objs = self._make_objects([s])
        return self.objs

    def step(self, action):
        """1ステップ進める。
        step_mode=frames : frameskip フレーム固定 (HRAの厳密なフレーム設定は未確定)
        step_mode=cell   : Ms. Pac-Man が次のマスに入るまで同じ行動を続ける (最大 cell_max_frames)。
                           固定フレームだと同じ(状態,行動)から毎回同じ次状態になるのは2割程度だが、
                           マス単位では過去の標本で85〜95%。全状態の決定性を保証する値ではない"""
        if self.step_mode != "cell":
            return self._substep(action, self.frameskip)
        start = self.objs.pacman if self.objs is not None else None
        raw = shaped = 0.0
        life_lost = level_cleared = False
        frames = 0
        while True:
            objs, r, sh, done, info = self._substep(action, self.cell_substep)
            raw += r
            shaped += sh
            frames += self.cell_substep
            life_lost |= info["life_lost"]
            level_cleared |= info["level_cleared"]
            moved = start is not None and objs.pacman is not None and objs.pacman != start
            if (done or life_lost or level_cleared or info["in_transition"] or moved
                    or start is None or frames >= self.cell_max_frames):
                break
        info.update(life_lost=life_lost, level_cleared=level_cleared)
        return objs, raw, shaped, done, info

    def _substep(self, action, nframes):
        a = self.action_set[action]
        raw = 0.0
        for k in range(nframes):
            raw += self.ale.act(a)
            self.frames += 1
            self.level_frames += 1
            if k >= nframes - self.ghost_frames:
                self._recent.append(self._screen())
            if self.ale.game_over():
                break
        if not self._recent:
            self._recent.append(self._screen())
        screens = list(self._recent)   # 直近 ghost_frames 枚 (ステップをまたいでも保持)
        lives = self.ale.lives()
        life_lost = lives < self.lives
        self.lives = lives
        done = self.ale.game_over()
        s = screens[-1]

        level_cleared = False
        if not self.in_transition and self.level_frames < SITE_SCAN_FRAMES:
            self.remaining |= self._add_sites(screens)
        if not self.in_transition:
            # 見えなくなったエサは、Ms. Pac-Man がそのマスの近くにいたときだけ「食べた」とみなす
            # (ゴーストや点滅で一時的に隠れたエサを消さないため)
            visible = self._visible_pellets(screens)
            pac = self._extract_ram()[0] if self.source == "ram" else self._extract_pixels(screens)[0]
            if pac is not None:
                self.remaining = {c for c in self.remaining
                                  if c in visible or max(abs(c[0] - pac[0]), abs(c[1] - pac[1])) > 1}
            if not self.remaining:
                level_cleared = True
                self.in_transition = True
                self.level += 1
        else:
            # 面クリア後: 壁の点滅が終わり、新しい迷路にエサが並んだら次の面を開始
            _, wall = self._colors(s)
            if wall is not None:
                lab, _ = ndimage.label(s == wall)
                n_small = sum(1 for sl in ndimage.find_objects(lab) if sl is not None and _is_small(sl))
                if n_small > 50:
                    self._register_maze(screens)
                    self.remaining = self._visible_pellets(screens)
                    self.in_transition = False
                    self.prev_pac = None
        if life_lost:
            self.prev_pac = None

        self.objs = self._make_objects(screens)
        shaped = raw + (self.ghost_penalty if life_lost else 0.0)
        info = {"frames": self.frames, "lives": lives, "life_lost": life_lost,
                "level_cleared": level_cleared, "level": self.level,
                "in_transition": self.in_transition}
        return self.objs, raw, shaped, done, info

    def compare_sources(self, screens=None):
        """RM-12: 画面抽出と RAM 読み取りの位置の差 (マス) を返す"""
        screens = screens or [self._screen()]
        pix_pac, pix_g = self._extract_pixels(screens)
        ram_pac, ram_g = self._extract_ram()
        return (pix_pac, ram_pac), list(zip(pix_g, ram_g))

    def close(self):
        self._gym.close()

In [ ]:
%%writefile hra/agent.py
"""
HRA 本体 (FR-12〜FR-37)

GVF (一般価値関数):
  迷路ごと・目標位置ごとに「その位置へ行く」ための擬似Q値 (0〜1) を表で持つ。
  状態 = Ms. Pac-Man の (位置, 向き)  [付録A.2]
  擬似報酬 = 目標位置に着いたら 1 (そこで終端 → 以後の価値は 0。FR-23)
  更新 = 全GVFを同じ遷移からオフポリシーで並行更新 (FR-20)
         式(8) 期待Sarsa (ランダム方策の価値) または 式(7) Q学習 (FR-22)

頭:
  各オブジェクトの頭 = オブジェクトのいる位置のGVF × 倍率 (FR-15)。画面にないオブジェクトは 0 (FR-16)

表の持ち方 (NFR-03):
  見つかった位置・状態だけに番号を振り (辞書)、その番号で引く numpy 配列に値を入れる。
  配列は見つかった分だけ伸ばすので、未到達の位置の分のメモリは使わない。
  付録A.2 の見積もり (4迷路 × 約400位置 × 約950状態 × 9行動 ≈ 14M要素) で float32 約55MB/γ。
"""
import numpy as np

from .env import N_ACTIONS, NORTH, EAST, SOUTH, WEST

# ALE の Ms. Pac-Man の行動: 0 NOOP 1 UP 2 RIGHT 3 LEFT 4 DOWN 5 UPRIGHT 6 UPLEFT 7 DOWNRIGHT 8 DOWNLEFT
# 向きごとに、その向きの成分を含む行動
HEADING_ACTIONS = {NORTH: [1, 5, 6], EAST: [2, 5, 7], SOUTH: [4, 7, 8], WEST: [3, 6, 8]}


class MazeTables:
    """1つの迷路の GVF 表とカウント"""

    def __init__(self, gammas):
        self.pos_index = {}      # 位置 → GVF番号 (FR-13: 新しい位置を見つけたら追加)
        self.state_index = {}    # (位置, 向き) → 状態番号
        self.state_pos = []      # 状態番号 → GVF番号 (その状態の位置)
        self.gammas = list(gammas)
        self.G = {g: np.zeros((16, 64, N_ACTIONS), dtype=np.float32) for g in self.gammas}
        self.n = np.zeros((64, N_ACTIONS), dtype=np.int64)   # カウントベース頭用 n(s,a)
        self._snap = {}

    @property
    def n_gvfs(self):
        return len(self.pos_index)

    def _grow(self):
        T, S = len(self.pos_index), len(self.state_index)

        def cap(old, need):
            return old if need <= old else max(old * 2, need)
        for g, arr in self.G.items():
            t_cap, s_cap, _ = arr.shape
            if T > t_cap or S > s_cap:
                new = np.zeros((cap(t_cap, T), cap(s_cap, S), N_ACTIONS), dtype=np.float32)
                new[:t_cap, :s_cap] = arr
                self.G[g] = new
        if S > self.n.shape[0]:
            new = np.zeros((cap(self.n.shape[0], S), N_ACTIONS), dtype=np.int64)
            new[:self.n.shape[0]] = self.n
            self.n = new

    def state(self, pos, direction):
        """状態番号を返す。初めての位置/状態なら GVF/状態を追加する"""
        if pos not in self.pos_index:
            self.pos_index[pos] = len(self.pos_index)
            self._snap.clear()
        key = (pos, direction)
        s = self.state_index.get(key)
        if s is None:
            s = len(self.state_index)
            self.state_index[key] = s
            self.state_pos.append(self.pos_index[pos])
        self._grow()
        return s

    def gvf_of(self, pos, radius):
        """オブジェクト位置に対応する GVF 番号。なければ radius マス以内で最も近い既知位置 (なければ -1)"""
        t = self.pos_index.get(pos)
        if t is not None:
            return t
        hit = self._snap.get(pos)
        if hit is None:
            hit, best = -1, None
            for dr in range(-radius, radius + 1):
                for dc in range(-radius, radius + 1):
                    q = self.pos_index.get((pos[0] + dr, pos[1] + dc))
                    d = abs(dr) + abs(dc)
                    if q is not None and (best is None or d < best):
                        hit, best = q, d
            self._snap[pos] = hit
        return hit

    def update(self, s, a, s_next, alpha, target):
        """1遷移で全GVFを更新 (FR-20)。s_next の位置を目標とするGVFは擬似報酬1で終端"""
        T = self.n_gvfs
        reached = self.state_pos[s_next]
        for g, G in self.G.items():
            nxt = G[:T, s_next, :]
            v = nxt.mean(axis=1) if target == "expected_sarsa" else nxt.max(axis=1)   # 式(8) / 式(7)
            y = g * v
            y[reached] = 1.0      # 擬似報酬1、終端なので以後の価値は0 (FR-23)
            if alpha == 1.0:
                G[:T, s, a] = y
            else:
                G[:T, s, a] += alpha * (y - G[:T, s, a])

    def trimmed(self):
        T, S = len(self.pos_index), len(self.state_index)
        return {
            "pos_index": self.pos_index, "state_index": self.state_index, "state_pos": self.state_pos,
            "G": {g: arr[:T, :S].copy() for g, arr in self.G.items()}, "n": self.n[:S].copy(),
        }

    @classmethod
    def from_trimmed(cls, d, gammas):
        m = cls(gammas)
        # 読み込み元 (チェックポイントの dict) を書き換えないようにコピーする
        m.pos_index, m.state_index, m.state_pos = dict(d["pos_index"]), dict(d["state_index"]), list(d["state_pos"])
        m._grow()
        T, S = len(m.pos_index), len(m.state_index)
        for g in m.gammas:
            # 保存時と γ が変わった場合は、最も近い γ の表を初期値として写す (途中で γ を変える実験用)
            src = g if g in d["G"] else min(d["G"], key=lambda k: abs(k - g))
            m.G[g][:T, :S] = d["G"][src]
        m.n[:S] = d["n"]
        return m


class HRAAgent:
    def __init__(self, cfg, seed=0):
        self.cfg = cfg
        gv, hd = cfg["gvf"], cfg["heads"]
        self.alpha = gv["alpha"]
        self.target = gv["target"]
        self.g_score = gv["gamma_score"]
        self.g_ghost = gv["gamma_ghost"]
        self.gammas = sorted({self.g_score, self.g_ghost})
        self.heads = hd
        self.snap = hd["snap_radius"]
        self.aggregator = cfg["aggregator"]["type"]
        self.normalize = cfg["aggregator"]["normalize"]
        ex = cfg["exploration"]
        self.use_div, self.div_max, self.div_steps = ex["diversification"], ex["div_max"], ex["div_steps"]
        self.use_count, self.kappa = ex["count_based"], ex["kappa"]
        self.heading_bonus = hd.get("heading_bonus", 0.0)
        em = cfg["executive_memory"]
        self.use_memory, self.memory_value = em["enabled"], em["value"]
        self.rng = np.random.default_rng(seed)
        self.mazes = {}
        self.N = 0                 # これまでに取った行動の総数 (式11)
        self.memory = {}           # 面番号 → 行動列 (FR-34)
        self.update_counts = True
        self.last_decision = None

    # ── 状態 ──
    def tables(self, maze):
        if maze not in self.mazes:
            self.mazes[maze] = MazeTables(self.gammas)
        return self.mazes[maze]

    def state(self, objs):
        if objs.pacman is None:
            return None
        return self.tables(objs.maze).state(objs.pacman, objs.direction)

    # ── 頭とアグリゲーター ──
    def _sum_heads(self, tab, s, positions, weights, gamma):
        idx, w = [], []
        for p, wt in zip(positions, weights):
            if p is None:
                continue                       # 画面にないオブジェクトは 0 (FR-16)
            t = tab.gvf_of(p, self.snap)
            if t >= 0:
                idx.append(t)
                w.append(wt)
        if not idx:
            return np.zeros(N_ACTIONS, dtype=np.float64)
        q = tab.G[gamma][np.array(idx), s, :]
        return (np.array(w, dtype=np.float64)[:, None] * q).sum(axis=0)

    def head_values(self, objs, s):
        """得点を生む頭の合計と、通常ゴーストの頭の合計 (倍率なし) を返す"""
        tab = self.tables(objs.maze)
        h = self.heads
        pos = objs.pellets + objs.power_pellets + [objs.fruit] + list(objs.blue_ghosts)
        wts = ([h["pellet"]] * len(objs.pellets) + [h["power_pellet"]] * len(objs.power_pellets)
               + [h["fruit"]] + [h["blue_ghost"]] * len(objs.blue_ghosts))
        score = self._sum_heads(tab, s, pos, wts, self.g_score)
        ghost = self._sum_heads(tab, s, objs.ghosts, [1.0] * len(objs.ghosts), self.g_ghost)
        return score, ghost

    def aggregate(self, score, ghost):
        if self.aggregator == "linear":                     # 式(5) FR-24
            return score + self.heads["ghost_linear"] * ghost
        # 正規化アグリゲーター FR-25
        if self.normalize == "minmax":
            lo, hi = score.min(), score.max()
            sn = (score - lo) / (hi - lo) if hi > lo else np.zeros_like(score)
        else:
            hi = score.max()
            sn = score / hi if hi > 0 else np.zeros_like(score)
        return sn + self.heads["ghost_normalized"] * ghost

    def count_head(self, tab, s):
        """式(11) Q_teh = κ sqrt(N^(1/4) / n(s,a))。n=0 は 1 として扱う"""
        n = np.maximum(tab.n[s], 1)
        return self.kappa * np.sqrt(self.N ** 0.25 / n)

    def q_values(self, objs, s, t, memory_action=None):
        q = np.zeros(N_ACTIONS, dtype=np.float64)
        if s is not None:
            q += self.aggregate(*self.head_values(objs, s))
            if self.use_count:
                q += self.count_head(self.tables(objs.maze), s)
            if self.heading_bonus:
                # 論文にない工夫: 進行方向を含む行動に小さなボーナス。
                # 左右に同じくらいの価値があるときの往復 (その場で止まる) を防ぐ
                q[HEADING_ACTIONS[objs.direction]] += self.heading_bonus
        if self.use_div and t < self.div_steps:            # 式(9)(10) FR-30/31
            q += self.rng.uniform(0.0, self.div_max, N_ACTIONS)
        if memory_action is not None:                       # FR-35
            q[memory_action] += self.memory_value
        return q

    def act(self, objs, s, t, memory_action=None):
        q = self.q_values(objs, s, t, memory_action)
        best = np.flatnonzero(q >= q.max() - 1e-12)
        a = int(self.rng.choice(best))                      # FR-27 (同点はランダム)
        self.last_decision = {'q': q.copy(), 'action': a, 'ties': len(best)}
        if s is not None and self.update_counts:
            self.tables(objs.maze).n[s, a] += 1
        if self.update_counts:
            self.N += 1
        return a

    # ── 学習 ──
    def learn(self, maze, s, a, s_next):
        if s is None or s_next is None:
            return
        self.tables(maze).update(s, a, s_next, self.alpha, self.target)

    # ── 実行記憶 (FR-34〜37) ──
    def memory_action(self, level, level_step, clean):
        if not self.use_memory or not clean:
            return None
        seq = self.memory.get(level)
        if seq is None or level_step >= len(seq):
            return None
        return seq[level_step]

    def remember(self, level, actions):
        if level not in self.memory or len(actions) < len(self.memory[level]):
            self.memory[level] = list(actions)

    # ── 集計・保存 ──
    def stats(self):
        return {"gvfs": sum(m.n_gvfs for m in self.mazes.values()),
                "states": sum(len(m.state_index) for m in self.mazes.values()),
                "mazes": len(self.mazes), "memory_levels": len(self.memory)}

    def state_dict(self):
        return {"mazes": {k: m.trimmed() for k, m in self.mazes.items()}, "N": self.N,
                "memory": self.memory, "rng": self.rng.bit_generator.state}

    def load_state_dict(self, d):
        self.mazes = {k: MazeTables.from_trimmed(v, self.gammas) for k, v in d["mazes"].items()}
        self.N = d["N"]
        self.memory = {k: list(v) for k, v in d["memory"].items()}
        self.rng.bit_generator.state = d["rng"]

In [ ]:
%%writefile hra/run.py
"""
HRA Ms. Pac-Man の学習・評価 (FR-40〜FR-44, NFR-01, NFR-02, RM-20, RM-21)

使い方:
  python -m hra.run train --config hra/config.yaml --out runs --sync /content/drive/MyDrive/HRA_runs
  python -m hra.run eval  --config hra/config.yaml --out runs --mode fixed --video
  python -m hra.run train --set aggregator.type=linear --set exploration.count_based=false

  --out  : 学習中に書き込む場所 (Colab ではローカルディスク /content/... を推奨)
  --sync : 定期的に成果物をコピーする場所 (Google Drive)。再開時は --out になければここから読む
"""
import argparse
import copy
import csv
import os
import pickle
import shutil
import time
import hashlib
from pathlib import Path

import numpy as np
import yaml

from .agent import HRAAgent
from .env import MsPacmanEnv

HERE = os.path.dirname(os.path.abspath(__file__))
CSV_FIELDS = ["episode", "score", "score_avg100", "frames", "steps", "level_reached", "levels_passed",
              "r_pellet", "r_power", "r_fruit", "r_blue_ghost", "r_ghost_contact",
              "gvfs", "states", "seconds"]


# ── 設定 (NFR-04) ──
def load_config(path=None, overrides=()):
    with open(path or os.path.join(HERE, "config.yaml"), encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    for ov in overrides:
        key, val = ov.split("=", 1)
        node = cfg
        parts = key.split(".")
        for p in parts[:-1]:
            node = node[p]
        node[parts[-1]] = yaml.safe_load(val)
    return cfg


def run_name(cfg):
    """FR-43: 設定ごとに結果を分ける"""
    if cfg["experiment"]["name"] != "auto":
        return cfg["experiment"]["name"]
    ex = cfg["exploration"]
    parts = [cfg["aggregator"]["type"],
             "div" if ex["diversification"] else "nodiv",
             "cnt" if ex["count_based"] else "nocnt",
             "eq8" if cfg["gvf"]["target"] == "expected_sarsa" else "eq7",
             cfg["env"].get("position_source", "pixels")]
    if cfg["env"].get("step_mode", "frames") == "cell":
        parts.append("cell")
    if cfg["executive_memory"]["enabled"]:
        parts.append("memory")
    return "_".join(parts)


# ── 報酬分解のログ (FR-11) ──
def classify_reward(raw, prev_objs):
    """1ステップの得点を、どのオブジェクトから得たかに分ける (ログ用)"""
    out = {"r_pellet": 0.0, "r_power": 0.0, "r_fruit": 0.0, "r_blue_ghost": 0.0}
    rest = raw
    if rest <= 0:
        return out
    # 1ステップ(8フレーム)で食べられるエサは高々数個なので、端数 (10/20/30...) をエサ、
    # 50 を含むならパワーエサ、残りの大きな値を青ゴースト/フルーツに割り当てる
    small = rest % 100
    if small >= 50 and prev_objs.power_pellets:
        out["r_power"] += 50
        small -= 50
    out["r_pellet"] += small
    rest -= out["r_power"] + out["r_pellet"]
    if rest > 0:
        if any(b is not None for b in prev_objs.blue_ghosts):
            out["r_blue_ghost"] += rest
        else:
            out["r_fruit"] += rest
    return out


# ── 1エピソード ──
def play_episode(env, agent, learn=True, noops=0, start_actions=None, frames_out=None,
                 observer=None, max_frames=None):
    objs = env.reset(noops=noops)
    if start_actions is not None:           # FR-41 人間の軌跡の途中から開始
        for a in start_actions:
            objs, _, _, done, _ = env.step(int(a))
            if done:
                objs = env.reset()
    rec = {k: 0.0 for k in ("score", "r_pellet", "r_power", "r_fruit", "r_blue_ghost", "r_ghost_contact")}
    t = 0
    level, level_step, clean, level_actions = 1, 0, True, []
    levels_passed = 0
    started = False     # 開始直後の待ち時間 (Ms. Pac-Man が動けない) はGVFを更新しない
    s = agent.state(objs)
    done = False
    while not done:
        mem_a = agent.memory_action(level, level_step, clean)
        a = agent.act(objs, s, t, mem_a)
        if observer is not None:
            observer.decision(env, agent, objs, s, t)
        before_frames = env.frames
        nxt, raw, shaped, done, info = env.step(a)
        if observer is not None:
            observer.observe(objs, nxt, raw, info, env.frames - before_frames)
        s_next = agent.state(nxt)
        if objs.pacman is not None and nxt.pacman is not None and nxt.pacman != objs.pacman:
            started = True
        valid = (started and not info["life_lost"] and not info["level_cleared"]
                 and not info["in_transition"] and nxt.maze == objs.maze)
        if learn and valid:
            agent.learn(objs.maze, s, a, s_next)
        # 記録
        rec["score"] += raw
        for k, v in classify_reward(raw, objs).items():
            rec[k] += v
        if info["life_lost"]:
            rec["r_ghost_contact"] += env.ghost_penalty
            clean = False
            started = False
        level_actions.append(a)
        level_step += 1
        if info["level_cleared"]:
            levels_passed += 1
            if clean and learn:
                agent.remember(level, level_actions)       # FR-34
            level, level_step, clean, level_actions = info["level"], 0, True, []
        if frames_out is not None:
            frames_out.append(env.ale.getScreenRGB())
        objs, s = nxt, s_next
        t += 1
        if max_frames is not None and env.frames >= max_frames and not done:
            rec['truncated'] = True
            break
    rec.update(frames=env.frames, steps=t, level_reached=level, levels_passed=levels_passed)
    if max_frames is not None:
        rec.setdefault('truncated', False)
    if observer is not None:
        rec.update(observer.summary())
    return rec


# ── 保存と再開 (NFR-01, RM-21) ──
def _atomic_pickle(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)


def sync_to(out_dir, sync_dir):
    """成果物 (チェックポイント・CSV・学習曲線・動画) を Google Drive などにコピー"""
    if not sync_dir:
        return
    os.makedirs(sync_dir, exist_ok=True)
    for name in os.listdir(out_dir):
        src = os.path.join(out_dir, name)
        if os.path.isfile(src) and not name.endswith(".tmp"):
            dst = os.path.join(sync_dir, name)
            shutil.copy2(src, dst + ".tmp")
            os.replace(dst + ".tmp", dst)


def restore_from(out_dir, sync_dir):
    """--out にチェックポイントがなく --sync にあれば、そちらから戻す"""
    ck = os.path.join(out_dir, "checkpoint.pkl")
    if os.path.exists(ck) or not sync_dir or not os.path.exists(os.path.join(sync_dir, "checkpoint.pkl")):
        return
    os.makedirs(out_dir, exist_ok=True)
    for name in os.listdir(sync_dir):
        src = os.path.join(sync_dir, name)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(out_dir, name))
    print(f"[再開] {sync_dir} から学習状態を復元しました")


def train(cfg, out_root, sync_root=None):
    name = run_name(cfg)
    out_dir = os.path.join(out_root, name)
    sync_dir = os.path.join(sync_root, name) if sync_root else None
    restore_from(out_dir, sync_dir)
    os.makedirs(out_dir, exist_ok=True)
    ex = cfg["experiment"]
    seed = ex["seed"]
    env = MsPacmanEnv(cfg, seed=seed)
    agent = HRAAgent(cfg, seed=seed)
    ck_path = os.path.join(out_dir, "checkpoint.pkl")
    csv_path = os.path.join(out_dir, "log.csv")
    start_ep, scores = 0, []
    if os.path.exists(ck_path):
        with open(ck_path, "rb") as f:
            ck = pickle.load(f)
        if 'config' in ck:
            for section in ('env', 'gvf', 'heads', 'aggregator', 'exploration', 'executive_memory'):
                if ck['config'][section] != cfg[section]:
                    raise ValueError(f'Resume configuration mismatch: {section}; use a separate experiment')
            if ck['config']['experiment']['seed'] != seed:
                raise ValueError('Resume seed mismatch')
        else:
            print('[注意] 旧モデルに設定記録がありません。過去の条件は未検証です。')
        agent.load_state_dict(ck["agent"])
        env.seed = ck["env_seed"]
        env.load_sites(ck["maze_colors"], ck["pellet_sites"])
        start_ep = ck["episode"]
        # チェックポイントより後のログ行 (やり直しになる分) を捨てる
        rows = []
        if os.path.exists(csv_path):
            with open(csv_path, newline="", encoding="utf-8") as f:
                rows = [r for r in csv.DictReader(f) if int(r["episode"]) <= start_ep]
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
            w.writeheader()
            w.writerows(rows)
        scores = [float(r["score"]) for r in rows]
        print(f"[再開] {start_ep} エピソード目から ({name})")
    else:
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()
        with open(os.path.join(out_dir, "config.yaml"), "w", encoding="utf-8") as f:
            yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
        print(f"[新規] {name}")

    def checkpoint(ep):
        _atomic_pickle({"agent": agent.state_dict(), "episode": ep, "env_seed": env.seed,
                        "config": copy.deepcopy(cfg),
                        "provenance": {'legacy_prefix_unverified': start_ep > 0 and
                                       ('config' not in ck or ck.get('provenance', {}).get('legacy_prefix_unverified', False)),
                                       'code_sha256': {n: hashlib.sha256((Path(HERE) / n).read_bytes()).hexdigest()
                                                       for n in ('agent.py', 'env.py', 'run.py')}},
                        "maze_colors": env.maze_colors, "pellet_sites": env.pellet_sites},
                       ck_path)
        from .plot import save_learning_curve
        save_learning_curve(csv_path, os.path.join(out_dir, "learning_curve.png"), title=name)
        sync_to(out_dir, sync_dir)

    for ep in range(start_ep + 1, ex["episodes"] + 1):
        t0 = time.time()
        rec = play_episode(env, agent, learn=True)
        scores.append(rec["score"])
        st = agent.stats()
        row = {k: rec.get(k, 0) for k in CSV_FIELDS}
        row.update(episode=ep, score_avg100=np.mean(scores[-100:]), gvfs=st["gvfs"], states=st["states"],
                   seconds=round(time.time() - t0, 2))
        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)
        if ep % ex["log_every"] == 0 or ep == 1:
            print(f"ep {ep:5d} | score {rec['score']:7.0f} | avg100 {row['score_avg100']:8.1f} | "
                  f"level {rec['level_reached']:3d} | frames {rec['frames']:6d} | GVF {st['gvfs']:4d} | "
                  f"{row['seconds']:.1f}s")
        if ep % ex["checkpoint_every"] == 0 or ep == ex["episodes"]:
            checkpoint(ep)
    env.close()
    return out_dir


def evaluate(cfg, out_root, mode="fixed", episodes=None, video=False, sync_root=None):
    """FR-40 固定開始 / FR-41 ランダム開始。GVFは更新しない"""
    name = run_name(cfg)
    out_dir = os.path.join(out_root, name)
    restore_from(out_dir, os.path.join(sync_root, name) if sync_root else None)
    with open(os.path.join(out_dir, "checkpoint.pkl"), "rb") as f:
        ck = pickle.load(f)
    ev = cfg["evaluation"]
    rng = np.random.default_rng(cfg["experiment"]["seed"] + 999)
    env = MsPacmanEnv(cfg, seed=cfg["experiment"]["seed"] + 10_000)
    env.load_sites(ck["maze_colors"], ck["pellet_sites"])
    agent = HRAAgent(cfg, seed=cfg["experiment"]["seed"] + 999)
    agent.load_state_dict(ck["agent"])
    human = None
    if mode == "random":
        if not ev["human_starts"]:
            raise SystemExit("ランダム開始評価には evaluation.human_starts (人間の行動列 .npy) が必要です (U-08)")
        human = np.load(ev["human_starts"])
    results = []
    for i in range(episodes or ev["episodes"]):
        frames = [] if (video and i == 0) else None
        if mode == "fixed":
            rec = play_episode(env, agent, learn=False, noops=int(rng.integers(0, ev["noop_max"] + 1)),
                               frames_out=frames)
        else:
            k = int(rng.integers(0, len(human)))
            rec = play_episode(env, agent, learn=False, start_actions=human[:k], frames_out=frames)
        results.append(rec)
        print(f"[{mode}] {i + 1}: score {rec['score']:.0f}  level {rec['level_reached']}")
        if frames:
            save_video(frames, os.path.join(out_dir, f"eval_{mode}_ep{ck['episode']}.mp4"))
    scores = [r["score"] for r in results]
    summary = {"mode": mode, "trained_episodes": ck["episode"], "mean": float(np.mean(scores)),
               "std": float(np.std(scores)), "max": float(np.max(scores)),
               "levels": [r["level_reached"] for r in results]}
    with open(os.path.join(out_dir, f"eval_{mode}.yaml"), "w", encoding="utf-8") as f:
        yaml.safe_dump(summary, f, allow_unicode=True)
    print(summary)
    sync_to(out_dir, os.path.join(sync_root, name) if sync_root else None)
    return summary


def save_video(frames, path, fps=15):
    try:
        import imageio.v2 as imageio
        imageio.mimsave(path, frames, fps=fps)
        print(f"動画保存: {path}")
    except Exception as e:  # imageio-ffmpeg がない環境など
        print(f"[動画スキップ] {e}")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("command", choices=["train", "eval"])
    p.add_argument("--config", default=None)
    p.add_argument("--set", action="append", default=[], help="設定の上書き 例: aggregator.type=linear")
    p.add_argument("--out", default="runs")
    p.add_argument("--sync", default=None, help="成果物を定期的にコピーする場所 (Google Drive)")
    p.add_argument("--mode", choices=["fixed", "random"], default="fixed")
    p.add_argument("--episodes", type=int, default=None)
    p.add_argument("--video", action="store_true")
    args = p.parse_args()
    cfg = load_config(args.config, args.set)
    if args.command == "train":
        if args.episodes:
            cfg["experiment"]["episodes"] = args.episodes
        train(cfg, args.out, args.sync)
    else:
        evaluate(cfg, args.out, args.mode, args.episodes, args.video, args.sync)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile hra/arcade.py
"""
アーケード版風のプレイ動画 (見た目だけ。学習・評価には使わない)

Atari 2600 版の画面から取り出したオブジェクトの位置をもとに、アーケード版 Pac-Man 風に描き直す:
黒背景・青い縁取りの迷路・点のエサ・点滅するパワーエサ・口の開閉するパックマン・目のあるゴースト。

  python -m hra.arcade --checkpoint runs/<設定名>/checkpoint.pkl --out play.mp4 --tries 8
    tries 回プレイして一番点数の高い回を録画する (実時間の速さ)
"""
import argparse
import pickle

import numpy as np
from PIL import Image, ImageDraw, ImageFont
from scipy import ndimage

from .env import (C_BLUE, C_GHOSTS, C_PACMAN, EAST, NORTH, RAM_GHOSTS, RAM_TO_PIX, SOUTH, WEST,
                  MsPacmanEnv, _center, _pack)

S = 4                    # 拡大率 (Atari の 1 画素 → S x S)
MAZE_ROWS = 172          # 迷路が描かれている行 (切り出し前の画面)
HEADER = 64
FOOTER = 44
BG = (0, 0, 0)
WALL = (33, 33, 222)
DOT = (255, 184, 151)
PAC = (255, 255, 0)
GHOST_RGB = [(255, 0, 0), (255, 184, 255), (0, 255, 255), (255, 184, 82)]   # 赤, ピンク, 水色, 橙
BLUE_GHOST = (33, 33, 255)
FLASH_GHOST = (222, 222, 255)
WHITE = (255, 255, 255)
MOUTH = {EAST: 0, SOUTH: 90, WEST: 180, NORTH: 270}


def _font(size):
    for name in ("arialbd.ttf", "DejaVuSans-Bold.ttf", "LiberationSans-Bold.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    try:
        return ImageFont.load_default(size=size)
    except TypeError:
        return ImageFont.load_default()


class ArcadeRenderer:
    def __init__(self, env, actor_mode='arcade'):
        self.env = env
        if actor_mode not in ('arcade', 'source'):
            raise ValueError('actor_mode must be arcade or source')
        self.actor_mode = actor_mode
        self.font = _font(22)
        self.small = _font(16)
        self.seen_color = [-99] * 4      # そのゴーストを通常色で最後に見たフレーム
        self.seen_blue = [-99] * 4       # 青/点滅で最後に見たフレーム
        self.blue_kind = [BLUE_GHOST] * 4
        self.tick = 0
        self._wall_key = None
        self._wall_img = None

    # ── 迷路 (壁の色の大きな塊 = 壁。小さな塊はエサなので除く) ──
    def _walls(self, s):
        key = (self.env.maze, self.env.wall)
        mask = s[:MAZE_ROWS] == self.env.wall
        lab, n = ndimage.label(mask)
        sizes = ndimage.sum(mask, lab, range(1, n + 1))
        big = np.isin(lab, 1 + np.flatnonzero(sizes > 16))
        if self._wall_key == key and self._wall_img is not None and big.sum() > 0.8 * self._wall_n:
            return self._wall_img          # 面の間は壁は変わらないので使い回す (点滅中は描き直さない)
        up = np.repeat(np.repeat(big, S, axis=0), S, axis=1)
        inner = ndimage.binary_erosion(up, iterations=2)
        outline = up & ~ndimage.binary_erosion(up)             # 外側の線
        outline2 = inner & ~ndimage.binary_erosion(inner)      # 内側の線 (二重線)
        img = np.zeros(up.shape + (3,), dtype=np.uint8)
        img[outline | outline2] = WALL
        self._wall_key, self._wall_img, self._wall_n = key, img, big.sum()
        return img

    def _ghost(self, d, cx, cy, color, eyes=True, pupil_dir=None):
        r = 3.6 * S
        d.pieslice([cx - r, cy - r, cx + r, cy + r], 180, 360, fill=color)
        d.rectangle([cx - r, cy, cx + r, cy + r * 0.8], fill=color)
        w = 2 * r / 3                                                  # 裾のギザギザ
        for k in range(3):
            x0 = cx - r + k * w
            d.polygon([(x0, cy + r * 0.8), (x0 + w / 2, cy + r * 1.05), (x0 + w, cy + r * 0.8)], fill=color)
        if eyes:
            dx, dy = {EAST: (1, 0), WEST: (-1, 0), NORTH: (0, -1), SOUTH: (0, 1)}.get(pupil_dir, (0, 0))
            for ex in (cx - r * 0.4, cx + r * 0.4):
                d.ellipse([ex - r * 0.3, cy - r * 0.45, ex + r * 0.3, cy + r * 0.15], fill=WHITE)
                px, py = ex + dx * r * 0.12, cy - r * 0.15 + dy * r * 0.12
                d.ellipse([px - r * 0.15, py - r * 0.15, px + r * 0.15, py + r * 0.15], fill=(33, 33, 255))
        else:                                                           # 青ゴーストの顔
            for ex in (cx - r * 0.35, cx + r * 0.35):
                d.rectangle([ex - r * 0.1, cy - r * 0.3, ex + r * 0.1, cy - r * 0.1], fill=DOT)
            pts = [(cx - r * 0.6 + i * r * 0.2, cy + r * (0.35 if i % 2 else 0.2)) for i in range(7)]
            d.line(pts, fill=DOT, width=max(1, S // 2))

    def _eyes(self, d, cx, cy, direction):
        r = 3.6 * S
        dx, dy = {EAST: (1, 0), WEST: (-1, 0), NORTH: (0, -1), SOUTH: (0, 1)}.get(direction, (0, 0))
        for ex in (cx - r * 0.4, cx + r * 0.4):
            d.ellipse([ex - r * 0.3, cy - r * 0.45, ex + r * 0.3, cy + r * 0.15], fill=WHITE)
            px, py = ex + dx * r * 0.12, cy - r * 0.15 + dy * r * 0.12
            d.ellipse([px - r * 0.15, py - r * 0.15, px + r * 0.15, py + r * 0.15], fill=(33, 33, 255))

    def frame(self, rgb, score, high, lives):
        env = self.env
        self.tick += 1
        s = _pack(rgb)
        W, H = 160 * S, MAZE_ROWS * S
        canvas = Image.new("RGB", (W, HEADER + H + FOOTER), BG)
        canvas.paste(Image.fromarray(self._walls(s)), (0, HEADER))
        d = ImageDraw.Draw(canvas)
        oy = HEADER
        top = env.crop_top

        # エサ・パワーエサ (追跡中の「残っているエサ」)
        sites = env.pellet_sites.get(env.maze, {})
        for c in env.remaining:
            y, x, power = sites[c]
            cy, cx = oy + (y + top + 0.5) * S, (x - 0.5) * S
            if power:
                if (self.tick // 12) % 2 == 0:
                    rr = 2.2 * S
                    d.ellipse([cx - rr, cy - rr, cx + rr, cy + rr], fill=DOT)
            else:
                rr = 0.55 * S
                d.rectangle([cx - rr, cy - rr, cx + rr, cy + rr], fill=DOT)

        actor_background = canvas.copy() if self.actor_mode == 'source' else None

        # フルーツ (既知の色以外の塊) → さくらんぼ
        known = [env.background, env.wall, C_PACMAN, 0, *C_GHOSTS, *C_BLUE]
        other = ~np.isin(s[:MAZE_ROWS], known)
        if other.sum() >= 6:
            fy, fx = _center(other)
            cx, cy = fx * S, oy + fy * S
            rr = 1.6 * S
            for ox, oyy in ((-1.3, 1.0), (1.3, 1.6)):
                d.ellipse([cx + ox * S - rr, cy + oyy * S - rr, cx + ox * S + rr, cy + oyy * S + rr], fill=(222, 0, 0))
            d.line([(cx - 1.3 * S, cy), (cx + 2 * S, cy - 3 * S), (cx + 1.3 * S, cy + 0.5 * S)], fill=(222, 151, 81), width=S // 2)

        # ゴースト: 位置は RAM から (画面では3フレームに1回しか描かれないため、毎フレーム描くには RAM が確実)。
        # 画面の色は「通常 / 青 / 点滅 / 食べられて目だけ」の判定にだけ使う
        ram = env.ale.getRAM()
        pos = [(int(ram[ay]) + RAM_TO_PIX[1], int(ram[ax]) + RAM_TO_PIX[0]) for ax, ay in RAM_GHOSTS]
        for i, col in enumerate(C_GHOSTS):
            c = _center(s[:MAZE_ROWS] == col)
            if c is not None and abs(c[0] - pos[i][0]) + abs(c[1] - pos[i][1]) < 12:
                self.seen_color[i] = self.tick
        for color_code, kind in ((C_BLUE[0], BLUE_GHOST), (C_BLUE[1], FLASH_GHOST)):
            mask = s[:MAZE_ROWS] == color_code
            if mask.any():
                lab, n = ndimage.label(mask)
                for by, bx in ndimage.center_of_mass(mask, lab, range(1, n + 1)):
                    dist = [abs(by - py) + abs(bx - px) for py, px in pos]
                    i = int(np.argmin(dist))
                    if dist[i] < 12:
                        self.seen_blue[i], self.blue_kind[i] = self.tick, kind
        any_ghost_visible = max(self.seen_color + self.seen_blue) >= self.tick - 6
        for i in range(4 if any_ghost_visible else 0):     # 死亡演出・面の切り替え中は描かない
            gy, gx = pos[i]
            if not (0 <= gy < MAZE_ROWS and 0 <= gx < 160):
                continue
            blue_recent = self.tick - self.seen_blue[i] <= 4
            color_recent = self.tick - self.seen_color[i] <= 6
            if blue_recent and self.seen_blue[i] >= self.seen_color[i]:
                self._ghost(d, gx * S, oy + gy * S, self.blue_kind[i], eyes=False)
            elif color_recent:
                self._ghost(d, gx * S, oy + gy * S, GHOST_RGB[i], pupil_dir=env.direction)
            elif self.tick > 12:
                self._eyes(d, gx * S, oy + gy * S, env.direction)     # 食べられて巣に戻る途中

        # パックマン (進行方向に口を開閉)
        c = _center(s[:MAZE_ROWS] == C_PACMAN)
        if c is not None:
            py, px = c
            cx, cy, r = px * S, oy + py * S, 3.8 * S
            mouth = abs((self.tick % 8) - 4) * 11                   # 0〜44 度
            base = MOUTH.get(env.direction, 0)
            d.pieslice([cx - r, cy - r, cx + r, cy + r], base + mouth, base + 360 - mouth, fill=PAC)

        # Diagnostic mode keeps exact source sprite pixels, including death animation.
        # Source multiplexing is intentional here: no invented positions or hitboxes.
        if actor_background is not None:
            canvas = actor_background
            pixels = np.asarray(canvas).copy()
            actor_mask = ~np.isin(s[:MAZE_ROWS], [env.background, env.wall, 0])
            up_mask = np.repeat(np.repeat(actor_mask, S, axis=0), S, axis=1)
            up_rgb = np.repeat(np.repeat(rgb[:MAZE_ROWS], S, axis=0), S, axis=1)
            area = pixels[HEADER:HEADER + H]
            area[up_mask] = up_rgb[up_mask]
            canvas = Image.fromarray(pixels)
            d = ImageDraw.Draw(canvas)

        # 見出しと残機
        d.text((W * 0.12, 6), "1UP", fill=WHITE, font=self.font)
        d.text((W * 0.12, 32), f"{int(score)}", fill=WHITE, font=self.font)
        d.text((W * 0.45, 6), "HIGH SCORE", fill=WHITE, font=self.font)
        d.text((W * 0.52, 32), f"{int(high)}", fill=WHITE, font=self.font)
        for k in range(max(0, lives - 1)):
            cx, cy, r = 24 + k * 34, HEADER + H + FOOTER / 2, 12
            d.pieslice([cx - r, cy - r, cx + r, cy + r], 180 + 35, 180 - 35 + 360, fill=PAC)
        d.text((W - 150, HEADER + H + 10), f"LEVEL {env.level}", fill=WHITE, font=self.small)
        return np.asarray(canvas)


class _Recorder:
    """ALE を包み、2 フレームごとにアーケード風の画面を動画に書き出す"""

    def __init__(self, env, writer, high):
        self._ale, self._env, self._w = env.ale, env, writer
        self.renderer = ArcadeRenderer(env)
        self.score, self.high, self._n = 0.0, high, 0

    def act(self, a):
        r = self._ale.act(a)
        self.score += r
        self.high = max(self.high, self.score)
        self._n += 1
        if self._n % 2 == 0 and hasattr(self._env, "remaining"):
            self._w.append_data(self.renderer.frame(self._ale.getScreenRGB(), self.score, self.high,
                                                    self._ale.lives()))
        return r

    def __getattr__(self, k):
        return getattr(self._ale, k)


def record(checkpoint, out, tries=8, config=None, fps=30):
    import imageio.v2 as imageio
    from .agent import HRAAgent
    from .run import load_config, play_episode
    cfg = load_config(config)
    with open(checkpoint, "rb") as f:
        ck = pickle.load(f)

    def run(i, writer=None, high=0):
        env = MsPacmanEnv(cfg, seed=i)
        env.load_sites(ck["maze_colors"], ck["pellet_sites"])
        ag = HRAAgent(cfg, seed=i)
        ag.load_state_dict(ck["agent"])
        ag.rng = np.random.default_rng(1000 + i)                 # 回ごとに違うプレイにする
        if writer is not None:
            env.ale = _Recorder(env, writer, high)
        return play_episode(env, ag, learn=False, noops=(7 * i) % 31)

    results = []
    for i in range(tries):
        r = run(i)
        results.append((r["score"], i))
        print(f"try {i}: score {r['score']:.0f}  level {r['level_reached']}", flush=True)
    best_score, best_i = max(results)
    writer = imageio.get_writer(out, fps=fps, macro_block_size=1, quality=7)
    r = run(best_i, writer, high=best_score)
    writer.close()
    print(f"保存: {out}  (score {r['score']:.0f}, level {r['level_reached']}, 学習 {ck['episode']} エピソード)")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--checkpoint", required=True)
    p.add_argument("--out", default="arcade_play.mp4")
    p.add_argument("--tries", type=int, default=8)
    p.add_argument("--config", default=None)
    a = p.parse_args()
    record(a.checkpoint, a.out, a.tries, a.config)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile hra/diagnose_collision.py
"""Record synchronized raw ALE / arcade frames without changing gameplay.

Explicit config is required: old checkpoints do not contain their configuration.
Life-loss markers report ALE's life counter, not the exact collision instant.
"""
import argparse
import csv
import hashlib
import json
import pickle
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
from PIL import Image, ImageDraw

from .agent import HRAAgent
from .arcade import ArcadeRenderer
from .env import MsPacmanEnv, RAM_PLAYER, RAM_GHOSTS, RAM_TO_PIX
from .run import load_config


class PairedRecorder:
    def __init__(self, env, writer, log, capture_start=0):
        self.ale = env.ale
        self.renderer = ArcadeRenderer(env, actor_mode='source')
        self.writer, self.log = writer, log
        self.frames = 0
        self.score = 0
        self.last_lives = self.ale.lives()
        self.events = []
        self.event_frame = -1000
        self.capture_start = capture_start

    def act(self, action):
        reward = self.ale.act(action)
        self.frames += 1
        self.score += reward
        lives = self.ale.lives()
        lost = lives < self.last_lives
        if lost:
            self.event_frame = self.frames
            self.events.append({'frame': self.frames, 'seconds': self.frames / 60,
                                'lives_before': self.last_lives, 'lives_after': lives})
        self.last_lives = lives
        ram = self.ale.getRAM()
        coords = [int(ram[a]) for pair in [RAM_PLAYER, *RAM_GHOSTS] for a in pair]
        self.log.writerow([self.frames, self.frames / 60, action, reward, self.score,
                           lives, int(lost), *coords])
        # No temporal frame blending: both panels come from this exact ALE frame.
        if self.frames % 2 == 0 and self.frames >= self.capture_start:
            raw = self.ale.getScreenRGB()
            arcade = self.renderer.frame(raw, self.score, self.score, lives)
            canvas = Image.new('RGB', (1280, 896))
            # Same maze scale AND origin as the reconstructed side (header=64).
            canvas.paste(Image.fromarray(raw[:172]).resize((640, 688), Image.Resampling.NEAREST), (0, 120))
            canvas.paste(Image.fromarray(arcade), (640, 56))
            d = ImageDraw.Draw(canvas)
            d.text((10, 8), 'RAW ALE (original pixels)', fill='white')
            d.text((650, 8), 'REDRAW WALLS / EXACT SOURCE ACTORS', fill='white')
            label = f'frame={self.frames}  t={self.frames / 60:.3f}s  lives={lives}  score={self.score}'
            d.text((10, 30), label, fill='white')
            if self.frames - self.event_frame < 120:
                d.text((650, 30), 'ALE LIFE COUNTER DECREASED (may lag contact)', fill='red')
            self.writer.append_data(np.asarray(canvas))
        return reward

    def __getattr__(self, name):
        return getattr(self.ale, name)


def diagnose(checkpoint, config, out, max_frames=12000, seed=0, focus_first_loss=False):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=False)  # Preserve previous diagnostic evidence.
    cfg = load_config(config)
    with open(checkpoint, 'rb') as f:
        ck = pickle.load(f)
    def prepare():
        env = MsPacmanEnv(cfg, seed=seed)
        env.load_sites(ck['maze_colors'], ck['pellet_sites'])
        ag = HRAAgent(cfg, seed=seed)
        ag.load_state_dict(ck['agent'])
        ag.rng = np.random.default_rng(seed + 1000)
        return env, ag, env.reset()

    capture_start = 0
    if focus_first_loss:
        probe, agent, obj = prepare()
        try:
            t = 0
            while probe.frames < max_frames:
                a = agent.act(obj, agent.state(obj), t)
                obj, _, _, done, info = probe.step(a)
                t += 1
                if info['life_lost']:
                    capture_start = max(0, probe.frames - 240)
                    max_frames = min(max_frames, probe.frames + 120)
                    print(f'First loss near frame {probe.frames}; recording from {capture_start}', flush=True)
                    break
                if done:
                    break
        finally:
            probe.close()
    env, ag, objs = prepare()
    try:
        with open(out / 'frames.csv', 'w', newline='', encoding='utf-8') as f, imageio.get_writer(
                str(out / 'comparison.mp4'), fps=30, macro_block_size=1, quality=7) as writer:
            log = csv.writer(f)
            log.writerow(['frame', 'seconds', 'ale_action', 'reward', 'score', 'lives', 'life_lost',
                          'pac_x', 'pac_y', 'red_x', 'red_y', 'pink_x', 'pink_y',
                          'cyan_x', 'cyan_y', 'orange_x', 'orange_y'])
            recorder = PairedRecorder(env, writer, log, capture_start)
            env.ale = recorder
            t = 0
            while recorder.frames < max_frames:
                action = ag.act(objs, ag.state(objs), t)
                objs, _, _, done, _ = env.step(action)
                t += 1
                if done:
                    break
        report = {'checkpoint': str(Path(checkpoint).resolve()), 'trained_episodes': ck['episode'],
                  'checkpoint_sha256': hashlib.sha256(Path(checkpoint).read_bytes()).hexdigest(),
                  'config': cfg, 'seed': seed, 'frames': recorder.frames, 'score': recorder.score,
                  'capture_start_frame': capture_start,
                  'actor_mode': 'source', 'panels_aligned': True,
                  'life_losses': recorder.events, 'ram_to_pixel_offset': RAM_TO_PIX,
                  'note': 'Diagnostic rollout, not benchmark evaluation. No GVF learning or checkpoint writes. '
                          'Life decrement can occur after collision animation. Display shapes are not hitboxes.'}
        (out / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
        print(json.dumps({k: report[k] for k in ('trained_episodes', 'frames', 'score', 'life_losses')}, indent=2))
    finally:
        env.close()


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', required=True)
    p.add_argument('--config', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--max-frames', type=int, default=12000)
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--focus-first-loss', action='store_true')
    a = p.parse_args()
    diagnose(a.checkpoint, a.config, a.out, a.max_frames, a.seed, a.focus_first_loss)

In [ ]:
%%writefile hra/review.py
"""Repeatable post-training evaluation followed by synchronized recording."""
import argparse
import csv
import hashlib
import importlib.metadata
import json
import pickle
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import yaml

from .agent import HRAAgent
from .env import MsPacmanEnv
from .run import load_config, play_episode
from .diagnose_collision import diagnose
from .behavior import BehaviorMetrics


def summarize(rows):
    scores = np.array([r['score'] for r in rows], dtype=float)
    means = np.random.default_rng(1729).choice(scores, (10000, len(scores))).mean(axis=1)
    return {'episodes': len(rows), 'mean': float(scores.mean()),
            'median': float(np.median(scores)), 'std': float(scores.std()),
            'min': float(scores.min()), 'max': float(scores.max()),
            'mean_bootstrap_95_ci': np.quantile(means, [.025, .975]).tolist(),
            'level_reached_rates': {str(k): float(np.mean([r['level_reached'] >= k for r in rows]))
                                    for k in range(2, max(r['level_reached'] for r in rows) + 1)}}


def review(checkpoint, config, out_root, episodes=30, seed=20260927, expected_episode=None,
           video=True, video_max_frames=30000, count_mode='episode', noop_max=None,
           max_frames=108000, diagnostics=False):
    if episodes < 1:
        raise ValueError('episodes must be positive')
    checkpoint = Path(checkpoint)
    cfg = load_config(config)
    if count_mode not in ('episode', 'frozen', 'carry'):
        raise ValueError('Invalid count_mode')
    noop_max = cfg['evaluation']['noop_max'] if noop_max is None else noop_max
    if noop_max < 0:
        raise ValueError('noop_max must be nonnegative')
    digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
    with checkpoint.open('rb') as f:
        ck = pickle.load(f)
    if expected_episode is not None and ck['episode'] != expected_episode:
        raise ValueError(f"Expected episode {expected_episode}, got {ck['episode']}")
    if 'config' in ck:
        for section in ('env', 'gvf', 'heads', 'aggregator', 'exploration', 'executive_memory'):
            if ck['config'][section] != cfg[section]:
                raise ValueError(f'Checkpoint configuration differs: {section}')
    out = Path(out_root) / datetime.now(timezone.utc).strftime('review_%Y%m%dT%H%M%S_%fZ')
    out.mkdir(parents=True, exist_ok=False)
    config_copy = out / 'effective_config.yaml'
    config_copy.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')
    metadata = {'checkpoint_sha256': digest, 'trained_episodes': ck['episode'], 'seed': seed,
                'config_provenance': 'embedded_verified' if 'config' in ck else 'explicit_runtime_config_legacy_checkpoint_unverified',
                'protocol': {'noop_max': noop_max, 'count_mode': count_mode,
                             'max_frames': max_frames, 'learning': False,
                             'exploration': 'configured heads retained'},
                'versions': {p: importlib.metadata.version(p) for p in ('numpy', 'gymnasium', 'ale-py')},
                'code_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                                for p in Path(__file__).parent.glob('*.py')}}
    (out / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print(f'REVIEW_DIR={out}', flush=True)
    rng = np.random.default_rng(seed)
    rows = []
    carried = None
    with (out / 'episodes.csv').open('w', newline='', encoding='utf-8') as f:
        writer = None
        for i in range(episodes):
            env_seed, agent_seed = seed + 10000 + i, seed + i
            env = MsPacmanEnv(cfg, seed=env_seed)
            try:
                env.load_sites(ck['maze_colors'], ck['pellet_sites'])
                agent = HRAAgent(cfg)
                agent.load_state_dict(ck['agent'])
                if carried is not None:
                    agent.load_state_dict(carried)
                agent.rng = np.random.default_rng(agent_seed)
                agent.update_counts = count_mode != 'frozen'
                noops = int(rng.integers(0, noop_max + 1))
                with ((out / f'trace_{i+1:03d}.jsonl').open('w', encoding='utf-8')
                      if diagnostics else nullcontext()) as trace:
                    extra = {'observer': BehaviorMetrics(trace), 'max_frames': max_frames} if diagnostics else {}
                    row = play_episode(env, agent, learn=False, noops=noops, **extra)
                if count_mode == 'carry':
                    carried = agent.state_dict()
                row.update(index=i + 1, env_seed=env_seed, agent_seed=agent_seed, noops=noops)
                if writer is None:
                    writer = csv.DictWriter(f, fieldnames=list(row))
                    writer.writeheader()
                writer.writerow(row)
                f.flush()
                rows.append(row)
                print(f"EVAL {i+1}/{episodes}: score={row['score']:.0f}, level={row['level_reached']}", flush=True)
            finally:
                env.close()
    summary = summarize(rows)
    summary['protocol'] = metadata['protocol']
    if diagnostics:
        summary['behavior_means'] = {k: float(np.mean([r[k] for r in rows])) for k in
            ('deaths', 'reversals_per_100_moves', 'stationary_frames', 'max_no_reward_frames')}
        summary['truncated_episodes'] = sum(bool(r['truncated']) for r in rows)
    summary.update(trained_episodes=ck['episode'], checkpoint_sha256=digest)
    (out / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print('SUMMARY ' + json.dumps(summary), flush=True)
    if video:
        diagnose(str(checkpoint), str(config_copy), str(out / 'synchronized_video'),
                 max_frames=video_max_frames, seed=seed, focus_first_loss=True)
    if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != digest:
        raise RuntimeError('Checkpoint changed during review')
    (out / 'COMPLETE.txt').write_text('Evaluation and requested recording completed. Checkpoint unchanged.\n', encoding='utf-8')
    print('COMPLETE ' + str(out), flush=True)
    return str(out)


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', required=True)
    p.add_argument('--config', required=True)
    p.add_argument('--out-root', required=True)
    p.add_argument('--episodes', type=int, default=30)
    p.add_argument('--seed', type=int, default=20260927)
    p.add_argument('--expected-episode', type=int)
    p.add_argument('--no-video', action='store_true')
    p.add_argument('--video-max-frames', type=int, default=30000)
    p.add_argument('--count-mode', choices=['episode', 'frozen', 'carry'], default='episode')
    p.add_argument('--noop-max', type=int)
    p.add_argument('--diagnostics', action='store_true')
    a = p.parse_args()
    review(a.checkpoint, a.config, a.out_root, a.episodes, a.seed, a.expected_episode,
           not a.no_video, a.video_max_frames, a.count_mode, a.noop_max, diagnostics=a.diagnostics)

In [ ]:
%%writefile hra/behavior.py
"""Passive metrics: reversals are A-B-A cell moves, not joystick changes."""
from collections import deque
import json


class BehaviorMetrics:
    def __init__(self, trace=None):
        self.path = deque(maxlen=3)
        self.moves = self.reversals = self.deaths = self.stall_frames = 0
        self.no_reward_frames = self.max_no_reward_frames = 0
        self.trace = trace
        self.recent = deque(maxlen=100)
        self.last_dump = -10000
        self.active = False

    def decision(self, env, agent, objs, state, step):
        if self.trace is None:
            return
        scores, ghosts = agent.head_values(objs, state) if state is not None else (None, None)
        d = agent.last_decision
        self.recent.append(dict(frame=env.frames, step=step, maze=objs.maze,
                                position=objs.pacman, remaining=len(objs.pellets) + len(objs.power_pellets),
                                action=d['action'], ties=d['ties'], q=d['q'].tolist(),
                                score_head=None if scores is None else scores.tolist(),
                                ghost_head=None if ghosts is None else ghosts.tolist()))

    def dump(self, reason):
        if self.trace is not None and self.recent:
            self.trace.write(json.dumps({'reason': reason, 'decisions': list(self.recent)}) + '\n')
            self.trace.flush()

    def observe(self, before, after, reward, info, frames):
        if info['life_lost']:
            self.deaths += 1
            self.dump('life_counter_decreased_not_exact_contact_time')
        valid = (not info['life_lost'] and not info['level_cleared']
                 and not info['in_transition'] and before.maze == after.maze
                 and before.pacman is not None and after.pacman is not None)
        if not valid:
            self.path.clear()
            self.no_reward_frames = 0
            self.active = False
            return
        if after.pacman != before.pacman:
            self.active = True
        if not self.active:
            return
        if not self.path:
            self.path.append(before.pacman)
        if after.pacman != before.pacman:
            self.moves += 1
            self.path.append(after.pacman)
            if len(self.path) == 3 and self.path[0] == self.path[2]:
                self.reversals += 1
        else:
            self.stall_frames += frames
        self.no_reward_frames = self.no_reward_frames + frames if reward == 0 else 0
        self.max_no_reward_frames = max(self.max_no_reward_frames, self.no_reward_frames)
        if self.no_reward_frames >= 600 and self.no_reward_frames - self.last_dump >= 600:
            self.dump('no_reward_600_frames')
            self.last_dump = self.no_reward_frames
        if reward:
            self.last_dump = -10000

    def summary(self):
        return dict(moves=self.moves, reversals=self.reversals, deaths=self.deaths,
                    reversals_per_100_moves=100 * self.reversals / max(1, self.moves),
                    stationary_frames=self.stall_frames,
                    max_no_reward_frames=self.max_no_reward_frames)

In [ ]:
%%writefile hra/audit_suite.py
"""Matched evaluation protocols and isolated clean-training configurations."""
import argparse
import csv
import copy
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import yaml

from .review import review
from .run import load_config


def suite(checkpoint, config, out_root, episodes=30, seed=730001):
    root = Path(out_root) / datetime.now(timezone.utc).strftime('audit_%Y%m%dT%H%M%S_%fZ')
    root.mkdir(parents=True)
    results = {}
    scores = {}
    # Same seeds across conditions. No policy tuning on this held-out batch.
    for label, counts, noops in [('baseline', 'episode', 30), ('no_noops', 'episode', 0),
                                 ('frozen_counts', 'frozen', 30), ('carry_counts', 'carry', 30)]:
        path = review(checkpoint, config, root / label, episodes=episodes, seed=seed,
                      video=label == 'baseline', count_mode=counts, noop_max=noops, diagnostics=True)
        results[label] = json.loads((Path(path) / 'summary.json').read_text())
        with (Path(path) / 'episodes.csv').open() as f:
            scores[label] = np.array([float(r['score']) for r in csv.DictReader(f)])
        if label != 'baseline':
            delta = scores[label] - scores['baseline']
            boot = np.random.default_rng(4401).choice(delta, (10000, len(delta))).mean(axis=1)
            results[label]['paired_difference_from_baseline'] = {
                'mean': float(delta.mean()), 'bootstrap95': np.quantile(boot, [.025, .975]).tolist()}
        (root / 'comparison.json').write_text(json.dumps(results, indent=2))
    (root / 'COMPLETE.txt').write_text('Four evaluation protocols finished. No model training performed.\n')
    return root


def clean_configs(config, out):
    """Explicit seeds/settings in names; no accidental resume from mixed legacy run."""
    root = Path(out)
    root.mkdir(parents=True, exist_ok=True)
    base = load_config(config)
    manifest = []
    for method in ('ram_cell', 'pixels_frames'):
        for seed in (101, 202, 303):
            cfg = copy.deepcopy(base)
            cfg['env']['position_source'] = 'ram' if method == 'ram_cell' else 'pixels'
            cfg['env']['step_mode'] = 'cell' if method == 'ram_cell' else 'frames'
            cfg['gvf']['alpha'] = 1.0
            cfg['experiment'].update(seed=seed, episodes=5000)
            identity = f'clean_{method}_alpha1_seed{seed}'
            cfg['experiment']['name'] = identity
            path = root / f'{identity}.yaml'
            text = yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False)
            if path.exists():                # 再実行しても同じ内容なら使い回す。違えば上書きせず止める
                if yaml.safe_load(path.read_text(encoding='utf-8')) != cfg:
                    raise ValueError(f'{path} already exists with different settings')
            else:
                path.write_text(text, encoding='utf-8')
            manifest.append({'config': path.as_posix(), 'name': identity, 'seed': seed,
                             'status': 'prepared_not_trained', 'episodes': 5000})
    (root / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    return manifest


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint')
    p.add_argument('--config', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--episodes', type=int, default=30)
    p.add_argument('--prepare-clean', action='store_true')
    a = p.parse_args()
    if a.prepare_clean:
        clean_configs(a.config, a.out)
    else:
        if not a.checkpoint:
            p.error('--checkpoint is required for evaluation')
        print(suite(a.checkpoint, a.config, a.out, a.episodes))

In [ ]:
%%writefile hra/plot.py
"""学習曲線 (FR-42, NFR-05): エピソードごとのスコアと100エピソード移動平均、フレーム数"""
import csv

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 論文 Table 1 の基準線
BASELINES = {"HRA (paper, fixed start)": 25304, "Human": 15693}


def _read(csv_path):
    with open(csv_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    ep = [int(r["episode"]) for r in rows]
    score = [float(r["score"]) for r in rows]
    avg = [float(r["score_avg100"]) for r in rows]
    frames = [int(r["frames"]) for r in rows]
    return ep, score, avg, frames


def learning_curve_figure(csv_paths, labels=None, baselines=True):
    """複数設定の比較もできる学習曲線 (FR-43)"""
    if isinstance(csv_paths, str):
        csv_paths = [csv_paths]
    labels = labels or [None] * len(csv_paths)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                                   gridspec_kw={"height_ratios": [3, 1]})
    for path, label in zip(csv_paths, labels):
        ep, score, avg, frames = _read(path)
        if not ep:
            continue
        if len(csv_paths) == 1:
            ax1.plot(ep, score, color="#9aa7b8", lw=0.6, alpha=0.6, label="score")
        ax1.plot(ep, avg, lw=2, label=(label or "") + " avg100")
        ax2.plot(ep, frames, lw=0.8, label=label)
    if baselines:
        for name, v in BASELINES.items():
            ax1.axhline(v, ls="--", lw=1, color="#888")
            ax1.text(ax1.get_xlim()[0], v, f" {name} {v:,}", va="bottom", fontsize=8, color="#555")
    ax1.set_ylabel("Score")
    ax1.legend(loc="upper left", fontsize=8)
    ax1.grid(alpha=0.3)
    ax2.set_ylabel("Frames / episode")
    ax2.set_xlabel("Episode")
    ax2.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def save_learning_curve(csv_path, png_path, title=None):
    fig = learning_curve_figure(csv_path)
    if title:
        fig.axes[0].set_title(title)
    fig.savefig(png_path, dpi=120)
    plt.close(fig)

## 評価と同期動画を作成

In [ ]:
from hra.review import review
RUN = 'normalized_div_cnt_eq8_ram_cell'
ROOT = f'/content/drive/MyDrive/HRA_runs/{RUN}'
RESULT = review(f'{ROOT}/checkpoint.pkl', 'hra/config.yaml', ROOT,
                episodes=30, expected_episode=11000, video=True, diagnostics=True)

In [ ]:
from IPython.display import Video, display
import json
from pathlib import Path
print(json.loads((Path(RESULT) / 'summary.json').read_text()))
display(Video(str(Path(RESULT) / 'synchronized_video/comparison.mp4'), embed=True, width=960))

## 評価条件の比較（各30回、学習は行わない）

In [ ]:
from hra.audit_suite import suite
AUDIT = suite(f'{ROOT}/checkpoint.pkl', 'hra/config.yaml', ROOT, episodes=30)
print(AUDIT)